# BookWise — Collaborative Filtering (ALS)
Dataset: GoodBooks-10k
Metode: Alternating Least Squares (implicit library)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.sparse import csr_matrix
import implicit
import joblib, os

DATA_DIR  = '../../data'
MODEL_DIR = '../model'

# GoodBooks-10k
ratings = pd.read_csv(f'{DATA_DIR}/ratings.csv')
ratings = ratings[ratings['rating'].between(1, 5)]

# Filter user & buku aktif
user_counts = ratings['user_id'].value_counts()
book_counts = ratings['book_id'].value_counts()
ratings = ratings[ratings['user_id'].isin(user_counts[user_counts >= 3].index)]
ratings = ratings[ratings['book_id'].isin(book_counts[book_counts >= 5].index)]

print(f'Filtered ratings: {len(ratings):,}')
print(f'Unique users: {ratings["user_id"].nunique():,}')
print(f'Unique books: {ratings["book_id"].nunique():,}')

In [ ]:
# Buat mapping index
unique_users = ratings['user_id'].unique()
unique_books = ratings['book_id'].unique()
user2idx = {u: i for i, u in enumerate(unique_users)}
book2idx = {b: i for i, b in enumerate(unique_books)}
idx2book = {i: b for b, i in book2idx.items()}

rows = ratings['book_id'].map(book2idx).values
cols = ratings['user_id'].map(user2idx).values
data = ratings['rating'].values.astype(np.float32)

# item-user matrix (implicit: items x users)
item_user_matrix = csr_matrix((data, (rows, cols)),
                               shape=(len(unique_books), len(unique_users)))
print(f'Matrix shape: {item_user_matrix.shape}')
sparsity = 1 - item_user_matrix.nnz / (item_user_matrix.shape[0] * item_user_matrix.shape[1])
print(f'Sparsity: {sparsity:.4%}')

In [ ]:
# Train ALS model
model = implicit.als.AlternatingLeastSquares(factors=50, iterations=20, regularization=0.1)
model.fit(item_user_matrix)
print('✅ ALS model trained.')
print(f'User factors shape: {model.user_factors.shape}')
print(f'Item factors shape: {model.item_factors.shape}')

In [ ]:
# Evaluasi RMSE & MAE pada test set
train_df, test_df = train_test_split(ratings, test_size=0.2, random_state=42)

n_users_model = model.user_factors.shape[0]
n_items_model = model.item_factors.shape[0]

y_true, y_pred = [], []
for _, row in test_df.iterrows():
    uid = int(row['user_id'])
    bid = int(row['book_id'])
    if uid in user2idx and bid in book2idx:
        u = user2idx[uid]
        b = book2idx[bid]
        if u < n_users_model and b < n_items_model:
            score = float(model.user_factors[u] @ model.item_factors[b])
            score = max(1.0, min(5.0, score))
            y_true.append(row['rating'])
            y_pred.append(score)

rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mae  = mean_absolute_error(y_true, y_pred)
print(f'RMSE: {rmse:.4f}')
print(f'MAE:  {mae:.4f}')
print(f'Test samples: {len(y_true):,}')

In [ ]:
# Visualisasi error distribution
errors = np.array(y_true) - np.array(y_pred)
plt.figure(figsize=(8, 4))
plt.hist(errors, bins=50, color='steelblue', edgecolor='white')
plt.xlabel('Prediction Error (true - pred)')
plt.ylabel('Frekuensi')
plt.title(f'Error Distribution ALS (RMSE={rmse:.4f}, MAE={mae:.4f})')
plt.axvline(0, color='red', linestyle='--')
plt.tight_layout()
plt.savefig('cf_error_distribution.png', dpi=150)
plt.show()

In [ ]:
# Simpan model
books_df = pd.read_pickle(f'{MODEL_DIR}/books.pkl')
cf_model = {
    'model':            model,
    'user2idx':         user2idx,
    'book2idx':         book2idx,
    'idx2book':         idx2book,
    'item_user_matrix': item_user_matrix,
    'unique_books':     unique_books,
}
joblib.dump(cf_model, f'{MODEL_DIR}/collaborative.pkl')
print('✅ Collaborative model saved.')